In [ ]:
# [Cell 0]
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/football-tracker")
else:
    DRIVE = Path("drive_checkpoints")
DRIVE.mkdir(parents=True, exist_ok=True)

MODELS_DIR = DRIVE / "models"
MODELS_DIR.mkdir(exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
print(f"Checkpoints: {DRIVE}")
print(f"Models: {MODELS_DIR}")

In [ ]:
# [Cell 1]
!pip install -q ultralytics huggingface_hub
from ultralytics import YOLO, settings
settings.update({"runs_dir": str(DRIVE / "runs")})
import yaml, json, cv2, os, numpy as np
from collections import defaultdict
print("Ready.")

In [ ]:
# [Cell 2]
import requests, zipfile, io

URLS = {
    "train": "https://huggingface.co/datasets/keremberke/football-object-detection/resolve/main/data/train.zip",
    "valid": "https://huggingface.co/datasets/keremberke/football-object-detection/resolve/main/data/valid.zip",
}

raw_dir = DATA_DIR / "football_raw"
raw_dir.mkdir(exist_ok=True)

for split, url in URLS.items():
    dest = raw_dir / split
    if (dest / "_annotations.coco.json").exists():
        print(f"{split}: already extracted")
        continue
    dest.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {split}...")
    r = requests.get(url, stream=True)
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        for member in z.namelist():
            if member.startswith("__MACOSX") or member.startswith("._"):
                continue
            target = dest / Path(member).name
            if not member.endswith("/") and not target.exists():
                z.extract(member, dest)
                src_file = dest / member
                if src_file != target:
                    import shutil
                    shutil.move(str(src_file), str(target))
    print(f"{split}: {len(list(dest.iterdir()))} files")

In [ ]:
# [Cell 3]
def convert_coco_to_yolo_ball(coco_json, images_dir, output_dir):
    (output_dir / "images").mkdir(parents=True, exist_ok=True)
    (output_dir / "labels").mkdir(parents=True, exist_ok=True)

    with open(coco_json) as f:
        coco = json.load(f)

    cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
    file_to_anns = defaultdict(list)
    for ann in coco["annotations"]:
        if cat_id_to_name[ann["category_id"]] == "football":
            img_info = next(i for i in coco["images"] if i["id"] == ann["image_id"])
            file_to_anns[img_info["file_name"]].append((ann, img_info))

    count = 0
    for fname in os.listdir(str(images_dir)):
        if fname.startswith("._") or fname == "_annotations.coco.json":
            continue
        img_path = images_dir / fname
        if not img_path.suffix.lower() in (".jpg", ".jpeg", ".png"):
            continue
        frame = cv2.imread(str(img_path))
        if frame is None:
            continue
        stem = Path(fname).stem
        cv2.imwrite(str(output_dir / "images" / f"{stem}.jpg"), frame)
        anns = file_to_anns.get(fname, [])
        if not anns:
            continue
        lines = []
        h, w = frame.shape[:2]
        for ann, img_info in anns:
            x, y, bw, bh = ann["bbox"]
            cx = (x + bw / 2) / w
            cy = (y + bh / 2) / h
            cw = bw / w
            ch = bh / h
            lines.append(f"0 {cx:.6f} {cy:.6f} {cw:.6f} {ch:.6f}")
        with open(output_dir / "labels" / f"{stem}.txt", "w") as lf:
            lf.write("\n".join(lines))
        count += 1
    return count

yolo_dir = DATA_DIR / "ball_yolo"
total = 0
for split in ["train", "valid"]:
    src = raw_dir / split
    coco_json = src / "_annotations.coco.json"
    if coco_json.exists():
        n = convert_coco_to_yolo_ball(coco_json, src, yolo_dir / split)
        print(f"{split}: {n} ball frames")
        total += n
print(f"Total: {total} ball frames")

In [ ]:
# [Cell 4]
data_yaml = yolo_dir / "ball.yaml"
yaml.dump({
    "path": str(yolo_dir.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "nc": 1,
    "names": ["ball"],
}, open(data_yaml, "w"))
print(f"Created {data_yaml}")

In [ ]:
# [Cell 5]
model = YOLO("yolo26s.pt")
results = model.train(
    data=str(data_yaml),
    epochs=100,
    imgsz=1280,
    batch=8,
    name="football_ball",
    pretrained=True,
    device=0,
    save_period=10,
    degrees=2,
    translate=0.05,
    scale=0.3,
    fliplr=0.0,
    mosaic=0.5,
    erasing=0.2,
)
print(f"Best model: {results.save_dir}")

In [ ]:
# [Cell 6]
runs = sorted((DRIVE / "runs" / "detect").glob("football_ball*"), reverse=True)
best_pt = runs[0] / "weights" / "best.pt"
best_pt_onnx = best_pt.with_suffix(".onnx")
model = YOLO(str(best_pt))
model.export(format="onnx")
import shutil
shutil.copy(best_pt, MODELS_DIR / "ball_detector.pt")
if best_pt_onnx.exists():
    shutil.copy(best_pt_onnx, MODELS_DIR / "ball_detector.onnx")
for f in MODELS_DIR.iterdir():
    print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")
print("Saved to Drive.")